In [1]:
import sys
sys.path.append('..')

import torch
from src import models, data, lens, functional
from src.utils import experiment_utils
from baukit import Menu, show

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
mt = models.load_model("gptj", device=device, fp16=True)
print(f"dtype: {mt.model.dtype}, device: {mt.model.device}, memory: {mt.model.get_memory_footprint()}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


In [3]:
dataset = data.load_dataset()

relation_names = [r.name for r in dataset.relations]
# relation_options = Menu(choices = relation_names, value = relation_names)
# show(relation_options) # !caution: tested in a juputer-notebook. baukit visualizations are not supported in vscode.

In [4]:
relation_names

['characteristic gender',
 'univ degree gender',
 'name birthplace',
 'name gender',
 'name religion',
 'occupation age',
 'occupation gender',
 'fruit inside color',
 'fruit outside color',
 'object superclass',
 'substance phase of matter',
 'task person type',
 'task done by tool',
 'word sentiment',
 'work location',
 'city in country',
 'company CEO',
 'company hq',
 'country capital city',
 'country currency',
 'country language',
 'country largest city',
 'food from country',
 'landmark in country',
 'landmark on continent',
 'person lead singer of band',
 'person father',
 'person mother',
 'person native language',
 'person occupation',
 'person plays instrument',
 'person sport position',
 'plays pro sport',
 'person university',
 'pokemon evolution',
 'president birth year',
 'president election year',
 'product by company',
 'star constellation name',
 'superhero archnemesis',
 'superhero person',
 'adjective antonym',
 'adjective comparative',
 'adjective superlative',
 'v

In [5]:
# select one from the above
relation_name='adjective superlative'

# relation_name = relation_options.value
# if isinstance(relation_name, (list, tuple)):
#     relation_name = relation_name[0]

relation = dataset.filter(relation_names=[relation_name])[0]
print(f"{relation.name} -- {len(relation.samples)} samples")
print("------------------------------------------------------")

experiment_utils.set_seed(12345) # set seed to a constant value for sampling consistency
train, test = relation.split(5)
print("\n".join([sample.__str__() for sample in train.samples]))

adjective superlative -- 80 samples
------------------------------------------------------
quick -> quickest
rich -> richest
thick -> thickest
strong -> strongest
great -> greatest


In [10]:
################### hparams ###################
layer = 5
beta = 2.5
###############################################

In [11]:
from src.operators import JacobianIclMeanEstimator

estimator = JacobianIclMeanEstimator(
    mt = mt, 
    h_layer = layer,
    beta = beta
)
operator = estimator(
    relation.set(
        samples=train.samples, 
    )
)

# Checking $faithfulness$

In [12]:
test = functional.filter_relation_samples_based_on_provided_fewshots(
    mt=mt, test_relation=test, prompt_template=operator.prompt_template, batch_size=4
)

In [13]:
sample = test.samples[0]
print(sample)
operator(subject = sample.subject).predictions

angry -> angriest


[PredictedToken(token=' most', prob=0.3580285608768463),
 PredictedToken(token=' the', prob=0.3062380850315094),
 PredictedToken(token='\n', prob=0.11806529760360718),
 PredictedToken(token=' greatest', prob=0.04016968980431557),
 PredictedToken(token=' ', prob=0.019885439425706863)]

In [14]:
hs_and_zs = functional.compute_hs_and_zs(
    mt = mt,
    prompt_template = operator.prompt_template,
    subjects = [sample.subject],
    h_layer= operator.h_layer,
)
h = hs_and_zs.h_by_subj[sample.subject]

## Approximating LM computation $F$ as an affine transformation

### $$ F(\mathbf{s}, c_r) \approx \beta \, W_r \mathbf{s} + b_r $$

In [41]:
z = operator.beta * (operator.weight @ h) + operator.bias

lens.logit_lens(
    mt = mt,
    h = z,
    get_proba = True
)

([('\n', 0.252),
  (' ', 0.181),
  (' ...', 0.127),
  (' Buenos', 0.057),
  (' the', 0.039),
  ('...', 0.036),
  (' Bras', 0.02),
  ('\\', 0.016),
  (' (', 0.016),
  (' Rome', 0.015)],
 {})

In [17]:
correct = 0
wrong = 0
for sample in test.samples:
    predictions = operator(subject = sample.subject).predictions
    known_flag = functional.is_nontrivial_prefix(
        prediction=predictions[0].token, target=sample.object
    )
    print(f"{sample.subject=}, {sample.object=}, ", end="")
    print(f'predicted="{functional.format_whitespace(predictions[0].token)}", (p={predictions[0].prob}), known=({functional.get_tick_marker(known_flag)})')
    
    correct += known_flag
    wrong += not known_flag
    
faithfulness = correct/(correct + wrong)

print("------------------------------------------------------------")
print(f"Faithfulness (@1) = {faithfulness}")
print("------------------------------------------------------------")

sample.subject='angry', sample.object='angriest', predicted=" most", (p=0.3580285608768463), known=(✗)
sample.subject='bad', sample.object='worst', predicted=" most", (p=0.370369553565979), known=(✗)
sample.subject='big', sample.object='biggest', predicted=" biggest", (p=0.28859010338783264), known=(✓)
sample.subject='brave', sample.object='bravest', predicted="\n", (p=0.3491964042186737), known=(✗)
sample.subject='bright', sample.object='brightest', predicted="\n", (p=0.5233341455459595), known=(✗)
sample.subject='calm', sample.object='calmest', predicted="\n", (p=0.31641092896461487), known=(✗)
sample.subject='cheap', sample.object='cheapest', predicted=" most", (p=0.28659385442733765), known=(✗)
sample.subject='clean', sample.object='cleanest', predicted=" most", (p=0.37529256939888), known=(✗)
sample.subject='cold', sample.object='coldest', predicted=" most", (p=0.4509657025337219), known=(✗)
sample.subject='crazy', sample.object='craziest', predicted=" the", (p=0.32000571489334106

# $causality$

In [18]:
################### hparams ###################
rank = 100
###############################################

In [19]:
experiment_utils.set_seed(12345) # set seed to a constant value for sampling consistency
test_targets = functional.random_edit_targets(test.samples)

## setup

In [23]:
source = test.samples[-2]
target = test_targets[source]

f"Changing the mapping ({source}) to ({source.subject} -> {target.object})"

'Changing the mapping (young -> youngest) to (young -> cheapest)'

### Calculate $\Delta \mathbf{s}$ such that $\mathbf{s} + \Delta \mathbf{s} \approx \mathbf{s}'$

<p align="center">
    <img align="center" src="causality-crop.png" style="width:80%;"/>
</p>

Under the relation $r =\, $*plays the instrument*, and given the subject $s =\, $*Miles Davis*, the model will predict $o =\, $*trumpet* **(a)**; and given the subject $s' =\, $*Cat Stevens*, the model will now predict $o' =\, $*guiter* **(b)**. 

If the computation from $\mathbf{s}$ to $\mathbf{o}$ is well-approximated by $operator$ parameterized by $W_r$ and $b_r$ **(c)**, then $\Delta{\mathbf{s}}$ **(d)** should tell us the direction of change from $\mathbf{s}$ to $\mathbf{s}'$. Thus, $\tilde{\mathbf{s}}=\mathbf{s}+\Delta\mathbf{s}$ would be an approximation of $\mathbf{s}'$ and patching $\tilde{\mathbf{s}}$ in place of $\mathbf{s}$ should change the prediction to $o'$ = *guitar* 

In [24]:
def get_delta_s(
    operator, 
    source_subject, 
    target_subject,
    rank = 100,
    fix_latent_norm = None, # if set, will fix the norms of z_source and z_target
):
    w_p_inv = functional.low_rank_pinv(
        matrix = operator.weight,
        rank=rank,
    )
    hs_and_zs = functional.compute_hs_and_zs(
        mt = mt,
        prompt_template = operator.prompt_template,
        subjects = [source_subject, target_subject],
        h_layer= operator.h_layer,
        z_layer=-1,
    )

    z_source = hs_and_zs.z_by_subj[source_subject]
    z_target = hs_and_zs.z_by_subj[target_subject]
    
    z_source *= fix_latent_norm / z_source.norm() if fix_latent_norm is not None else 1.0
    z_target *= z_source.norm() / z_target.norm() if fix_latent_norm is not None else 1.0

    delta_s = w_p_inv @  (z_target.squeeze() - z_source.squeeze())

    return delta_s, hs_and_zs

delta_s, hs_and_zs = get_delta_s(
    operator = operator,
    source_subject = source.subject,
    target_subject = target.subject,
    rank = rank
)

In [25]:
import baukit

def get_intervention(h, int_layer, subj_idx):
    def edit_output(output, layer):
        if(layer != int_layer):
            return output
        functional.untuple(output)[:, subj_idx] = h 
        return output
    return edit_output

prompt = operator.prompt_template.format(source.subject)

h_index, inputs = functional.find_subject_token_index(
    mt=mt,
    prompt=prompt,
    subject=source.subject,
)

h_layer, z_layer = models.determine_layer_paths(model = mt, layers = [layer, -1])

with baukit.TraceDict(
    mt.model, layers = [h_layer, z_layer],
    edit_output=get_intervention(
#         h = hs_and_zs.h_by_subj[source.subject],         # let the computation proceed as usual
        h = hs_and_zs.h_by_subj[source.subject] + delta_s, # replace s with s + delta_s
        int_layer = h_layer, 
        subj_idx = h_index
    )
) as traces:
    outputs = mt.model(
        input_ids = inputs.input_ids,
        attention_mask = inputs.attention_mask,
    )

lens.interpret_logits(
    mt = mt, 
    logits = outputs.logits[0][-1], 
    get_proba=True
)

[(' quickest', 0.174),
 (' most', 0.136),
 (' cle', 0.136),
 (' clean', 0.074),
 (' cheapest', 0.055),
 (' shortest', 0.039),
 ('\n', 0.036),
 (' best', 0.032),
 (' least', 0.028),
 (' the', 0.026)]

## Measuring causality

In [27]:
from src.editors import LowRankPInvEditor

svd = torch.svd(operator.weight.float())
editor = LowRankPInvEditor(
    lre=operator,
    rank=rank,
    svd=svd,
)

In [28]:
# precomputing latents to speed things up
hs_and_zs = functional.compute_hs_and_zs(
    mt = mt,
    prompt_template = operator.prompt_template,
    subjects = [sample.subject for sample in test.samples],
    h_layer= operator.h_layer,
    z_layer=-1,
    batch_size = 2
)

success = 0
fails = 0

for sample in test.samples:
    target = test_targets.get(sample)
    assert target is not None
    edit_result = editor(
        subject = sample.subject,
        target = target.subject
    )
    
    success_flag = functional.is_nontrivial_prefix(
        prediction=edit_result.predicted_tokens[0].token, target=target.object
    )
    
    print(f"Mapping {sample.subject} -> {target.object} | edit result={edit_result.predicted_tokens[0]} | success=({functional.get_tick_marker(success_flag)})")
    
    success += success_flag
    fails += not success_flag
    
causality = success / (success + fails)

print("------------------------------------------------------------")
print(f"Causality (@1) = {causality}")
print("------------------------------------------------------------")

Mapping angry -> simplest | edit result= shortest (p=0.337) | success=(✗)
Mapping bad -> biggest | edit result= biggest (p=0.823) | success=(✓)
Mapping big -> lowest | edit result= lowest (p=0.772) | success=(✓)
Mapping brave -> poorest | edit result= most (p=0.278) | success=(✗)
Mapping bright -> happiest | edit result= happiest (p=0.961) | success=(✓)
Mapping calm -> lightest | edit result= longest (p=0.426) | success=(✗)
Mapping cheap -> zestiest | edit result= most (p=0.351) | success=(✗)
Mapping clean -> smallest | edit result= smallest (p=0.956) | success=(✓)
Mapping cold -> fullest | edit result= fullest (p=0.660) | success=(✓)
Mapping crazy -> poorest | edit result= most (p=0.283) | success=(✗)
Mapping cruel -> easiest | edit result= easiest (p=0.701) | success=(✓)
Mapping dark -> smallest | edit result= smallest (p=0.856) | success=(✓)
Mapping deep -> laziest | edit result= most (p=0.202) | success=(✗)
Mapping dirty -> youngest | edit result= youngest (p=0.787) | success=(✓)
M